#### **Install Dependencies**

In [1]:
! pip install -q pylate torchvision

#### **Load Dataset**

In [2]:
from datasets import load_dataset

# load Amharic Question Answering dataset
amharic_qa = load_dataset("israel/AmharicQA", split="validation").shuffle(seed=42)
amharic_qa

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


Dataset({
    features: ['context', 'question', 'answers'],
    num_rows: 595
})

In [3]:
queries = amharic_qa['question']
docs = amharic_qa['context']
answers = amharic_qa['answers']

len(queries), len(docs), len(answers)

(595, 595, 595)

#### **Build the PLAID Index with Amharic ColBERT**

In [ ]:
from pylate import indexes, models, retrieve

# Step 1: Load the ColBERT model
model = models.ColBERT(
    model_name_or_path="rasyosef/colbert-amharic-base",
)

# Step 2: Initialize the Voyager index
index = indexes.PLAID(
    index_folder="pylate-index",
    index_name="index",
    override=True,  # This overwrites the existing index if any
)

In [ ]:
# Step 3: Encode the documents
documents_ids, documents, ids_to_doc = [], [], {}
for idx, document in enumerate(list(set(docs))):
    documents_ids.append(str(idx))
    documents.append(document)
    ids_to_doc[str(idx)] = document

documents_embeddings = model.encode(
    documents,
    batch_size=32,
    is_query=False,  # Ensure that it is set to False to indicate that these are documents, not queries
    show_progress_bar=True,
)

In [6]:
# Step 4: Add document embeddings to the index by providing embeddings and corresponding ids
index.add_documents(
    documents_ids=documents_ids,
    documents_embeddings=documents_embeddings,
)

Computing centroids of embeddings.
Creating FastPlaid index.


#### **Query the Index**

In [7]:
idx = 6
query = queries[idx]
print(query)

በአጼ ቀዳማዊ ኃይለ ሥላሴ ታውጆ እስከ 1948 አ.ም. ሲያገለግል የነበረው ሕገ መንግሥት መች ነበር የታወጀው?


In [12]:
# Step 1: Initialize the ColBERT retriever
retriever = retrieve.ColBERT(index=index)

# Step 2: Encode the queries
queries_embeddings = model.encode(
    [query],
    batch_size=32,
    is_query=True,
    show_progress_bar=False,
)

# Step 3: Retrieve top-k documents
scores = retriever.retrieve(
    queries_embeddings=queries_embeddings,
    k=3,  # Retrieve the top 10 matches for each query
)

scores

[[{'id': '25', 'score': 24.0615234375},
  {'id': '40', 'score': 19.552734375},
  {'id': '45', 'score': 19.203857421875}]]

In [13]:
print("Top Results:")
for score in scores[0]:
    print({'score': score['score'], 'text': ids_to_doc[score['id']]})

Top Results:
{'score': 24.0615234375, 'text': 'በ1240 አካባቢ በግብጽ አገር ቅብጡ አቡል ፋዳዒል እብን አል-አሣል ፍትሐ ነገሥት በአረብኛ ጻፉ። እብን አል-አሣል ሕጎቹን የወሰዱት በከፊል ከሃዋርያት ጽህፈቶችና ከሕገ ሙሴ፤ በከፊልም ከድሮ ቢዛንታይን ነገስታት ሕጎች ነበር። መጽሐፉ በግዕዝ ተተርጉሞ ወደ ኢትዮጵያ የገባበት ወቅት በዐፄ ዘርዐ ያዕቆብ ዘመን በ1450 አከባቢ እንደነበር የሚል ታሪካዊ መዝገብ አለ።  ቢሆንም መጀመርያ እንደ ሕገ ምንግሥት መጠቀሙን የተመዘገበው በዓፄ ሠርፀ ድንግል ዘመን (ከ1555 ጀምሮ) ነው። ፍትሐ ነገስት እስከ 1923 ዓ.ም. የብሄሩን ዋና ሕግ ሆኖ ቆየ። በ1904 ዓ.ም. ዐጼ 2 ምኒልክ የዘመናዊ ሕገ መንግሥት ፅንሰ ሀሳብ ተረድተው በጸሀፊው መምሬ ብስራት አንድ ሕገ መንግሥት የሚመስል ሰነድ ታተመ፤ ይህ ግን በውነት ሕገ መንግሥት አይባልም። «በምኒሊክ ስለተቋቋሙት ሚኒስቴሮች በምሳሌ የቀረበ ጽሑፍ» በምሳሌና በትርጓሜ የእያንዳንዱን ሚኒስቴር ሥራ እንደ ሰውነት አካላት አስመሰለ። ለምሳሌ፦ ኢትዮጵያ መጀመርያ ዘመናዊ ሕገ መንግሥት የተቀበለችው በ1923 አመተ ምኅረት በአጼ ቀዳማዊ ኃይለ ሥላሴ የታወጀ ሲሆን የሕግ አማካሪ ቤቶች በሁለት ያስከፈለ ነው።  ይህ የመንግሥታቸው ዋና መሠረት ሆኖ እስከ 1948 አ.ም. አገለገለ፤ በዚያ አመት ተሻሽሎ በወጣ ሕገ መንግሥት ሕዝቦች በመንግሥት ሥራ የሚጫወቱት ሚና እንደገና ተስፋፋ።  ይህ ብቻ የአገሪቱ ሕገ መንግሥት ሆኖ እስከ 1967 አ.ም. ቆየ፤ የዛኔ በደርግ (በሕገ መንግስት እራሱ ባልሆነ ሂደት) ተሰረዘ።'}
{'score': 19.552734375, 'text': 'የሕገ መንግሥት ታሪክ የሱመር ከተማ ላጋሽ አለቃ ኡሩካጊና በ2093 ዓክልበ. ያሕል ያዋጀ ሕገ ፍትሕ 